READING PDF

In [33]:
from pypdf import PdfReader

#Loading the PDF

reader = PdfReader("/home/nineleaps/Documents/Python Files/sample_rag_dataset.pdf")

print("Number of pages in the document : ", len(reader.pages))

#Reading text from all the pages

text = ''

for page in reader.pages:
    text += page.extract_text()

print(text)

print("Text Extracted Succesfully!")

Number of pages in the document :  2
Sample PDF for Mini RAG Chatbot Assignment
This PDF is created as a sample dataset for a Retrieval-Augmented Generation (RAG) chatbot project.
The chatbot can ingest this document, split it into chunks, create embeddings, store them in a vector
database, and answer user questions based on the content.
What is RAG?
Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with
large language models. Instead of relying only on pretrained knowledge, the chatbot retrieves relevant
documents from an external knowledge base and uses them to generate accurate responses.
Components of a RAG System
A basic RAG system contains document loading, text chunking, embedding generation, vector storage,
retrieval, prompt construction, and response generation using an LLM.
Embeddings
Embeddings are numerical vector representations of text. Similar texts have similar embeddings.
Popular embedding models include OpenAI embeddings an

CHUNKING (USING lANGCHAIN)

In [34]:
#checking for text splitter working

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Working Successfully")

Working Successfully


In [35]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 50)

chunks = splitter.split_text(text)

print(chunks)

print(chunks[0])

['Sample PDF for Mini RAG Chatbot Assignment', 'This PDF is created as a sample dataset for a Retrieval-Augmented Generation (RAG) chatbot project.', 'The chatbot can ingest this document, split it into chunks, create embeddings, store them in a', 'into chunks, create embeddings, store them in a vector', 'database, and answer user questions based on the content.\nWhat is RAG?', 'Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with', 'large language models. Instead of relying only on pretrained knowledge, the chatbot retrieves', 'on pretrained knowledge, the chatbot retrieves relevant', 'documents from an external knowledge base and uses them to generate accurate responses.', 'Components of a RAG System', 'A basic RAG system contains document loading, text chunking, embedding generation, vector storage,', 'retrieval, prompt construction, and response generation using an LLM.\nEmbeddings', 'Embeddings are numerical vector representations of 

EMBEDDINGS

In [ ]:
from sentence_transformers import SentenceTransformer

#choosing a model for creating embeddings

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

print(embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5041.58it/s]


(35, 384)


STORING IN VECTOR DATABASE

In [37]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

CONVERTING USER QUESTION INTO EMBEDDINGS

In [43]:
query = 'What is embedding?'

query_embedding = model.encode([query])

RETRIEVING SIMILAR CHUNKS

In [44]:
k = 3

distances, indices = index.search(
    np.array(query_embedding),
    k
)

print(indices)

[[12  3 13]]


GETTING RELEVANT CONTEXT

In [45]:
retrieved_chunks = [chunks[i] for i in indices[0]]

context = "\n".join(retrieved_chunks)

print(context)

Embeddings are numerical vector representations of text. Similar texts have similar embeddings.
into chunks, create embeddings, store them in a vector
Popular embedding models include OpenAI embeddings and Sentence Transformers.
Vector Databases


CONFIGURING THE GEMINI LLM

In [46]:
#using gemini AI

import google.generativeai as genai

#Configuring API

genai.configure(api_key = "AQ.Ab8RN6KzERnZAsPzRO122J41A0Z4bOXDIIh15YeyhOipFTCL0Q")

model_gemini = genai.GenerativeModel("gemini-2.5-flash")

response = model_gemini.generate_content("Hello")

print(response.text)

Hello! How can I help you today?


SENDING THE CONTEXT TO LLM

In [48]:
#Creating a prompt for Gemini

prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.

Give the Answer in detail.

If the answer is not present in the context,
say:
'I could not find the answer in the document.'

Context:
{context}

Question:
{query}
"""

#Generating response
response = model_gemini.generate_content(
    prompt
)

#Final answer
print(response.text)

Embeddings are numerical vector representations of text. Similar texts have similar embeddings.
